# Gridvia Analytics — Pipeline day-ahead
Notebook orquestador. **No contiene lógica**: solo coordina las clases de `src/`.
Toda la maquinaria (descarga, limpieza, modelado, evaluación) vive en módulos versionables.

| Fase | Clase responsable | Celdas originales |
|---|---|---|
| Descarga | COESDownloader, ERA5Downloader, FichaDownloader | 6-21, 30 |
| Consolidación | PlantConsolidator, PhysicalCleaner, TemporalValidator, StaticFeatureBuilder | 12-27, 31-32 |
| Modelado | WalkForwardRunner (+ SmartPersistence, PearsonR) | 33-41 |
| Evaluación | TaylorDiagram, ShapAnalyzer, MetricsReporter | 43-64 |

## Opción A — flujo completo encadenado

In [ ]:
from src.pipeline import GridviaPipeline

pipe = (GridviaPipeline("conf/config.yaml")
        .descargar()
        .consolidar()
        .modelar()
        .evaluar())

## Opción B — fase por fase (para depurar)
Útil mientras migras el cuerpo de cada método. Permite inspeccionar el estado entre fases.

In [ ]:
from src.pipeline import GridviaPipeline

pipe = GridviaPipeline("conf/config.yaml")

In [ ]:
pipe.descargar()
print("COES:", None if pipe.coes is None else pipe.coes.shape)
print("ERA5:", None if pipe.era5 is None else pipe.era5.shape)

In [ ]:
pipe.consolidar()
print("largo_unificado:", None if pipe.largo_unificado is None else pipe.largo_unificado.shape)
print("df_static:", None if pipe.df_static is None else pipe.df_static.shape)

In [ ]:
pipe.modelar()

In [ ]:
pipe.evaluar()

## Experimento exp02 (Solcast 15min)
Mismo pipeline, distinta resolución. COES NO se resamplea; se suma Solcast 15min.
```python
# en conf/config.yaml -> resolucion.coes_salida: "15min"
# pipe.descargar() usará COESDownloader(output_resolution="15min") + SolcastDownloader
```